In [1]:
import ee 
import geemap
from IPython.display import display
import geopandas as gpd
import pandas as pd
import numpy as np
from datetime import datetime, UTC



In [2]:
ee.Initialize()

In [3]:
nm_path = r"C:\Users\bsf31\Documents\data\NM\nm_vector.gpkg"
gdf = gpd.read_file(nm_path, layer='lordsburg_parks')

In [4]:
gdf

,id,area,place,canopy_min,canopy_max,canopy_mean,canopy_std,canopy_median,canopy_area,understory_area,understory_percent,overstory_area,overstory_percent,canopy_cover_percent,geometry
0,0,4.747221e+06,Short Park,3.28084,55.77428,8.083212,6.721449,6.56168,1.960377e+06,6384.0,0.134433,10536.0,0.221865,0.356419,"MULTIPOLYGON (((714938.12 3581667.2, 714672.68..."
1,0,4.747221e+06,Gold Street Park,3.28084,55.77428,8.083212,6.721449,6.56168,1.960377e+06,6384.0,0.134433,10536.0,0.221865,0.356419,"MULTIPOLYGON (((716160.457 3581684.614, 716181..."


In [5]:
ee_parks = geemap.geopandas_to_ee(gdf).geometry()

In [13]:
# Define time range for NDVI calculation
start_date = '2024-09-01'
end_date = '2024-09-13'

# Load Sentinel-2 collection and filter
sentinel2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(ee_parks) \
    .filterDate(start_date, end_date) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))  

        

In [7]:
def describe_image_collection(ic, max_images=5):
    info = ic.limit(max_images).getInfo()
    images = info.get('features', [])
    
    print(f"ImageCollection ID: {info.get('id', 'N/A')}")
    print(f"Total Images Shown: {len(images)}\n")
    
    for i, img in enumerate(images):
        props = img.get('properties', {})
        timestamp = props.get('system:time_start', None)
        date_str = "N/A"
        if timestamp:
            # Use timezone-aware UTC object (Python 3.12+ compatible)
            date_str = datetime.fromtimestamp(timestamp / 1000, tz=UTC).strftime('%Y-%m-%d %H:%M:%S %Z')
        
        cloud_cover = props.get('CLOUDY_PIXEL_PERCENTAGE',
                        props.get('system:cloud_coverage', 'N/A'))
        if isinstance(cloud_cover, float):
            cloud_cover = f"{cloud_cover:.2f}%"
        
        print(f"Image {i+1}:")
        print(f"  ID: {img.get('id', 'N/A')}")
        print(f"  Date: {date_str}")
        print(f"  Cloud Cover: {cloud_cover}")
        print(f"  Bands: {[band['id'] for band in img.get('bands', [])]}")
        print(f"  CRS: {img.get('bands', [{}])[0].get('crs', 'N/A')}")
        print()


In [17]:
describe_image_collection(sentinel2)


ImageCollection ID: COPERNICUS/S2_SR_HARMONIZED
Total Images Shown: 1

Image 1:
  ID: COPERNICUS/S2_SR_HARMONIZED/20240912T174931_20240912T180223_T12SYA
  Date: 2024-09-12 18:05:12 UTC
  Cloud Cover: 0.00%
  Bands: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']
  CRS: EPSG:32612



In [28]:
Map = geemap.Map()
# Limit number of images to visualize
image_list = sentinel2.limit(4).toList(4)

for i in range(1):
    image = ee.Image(image_list.get(i)).clip(ee_parks)
    props = image.getInfo().get("properties", {})
    
    # Format date
    timestamp = props.get("system:time_start")
    if timestamp:
        date_str = datetime.fromtimestamp(timestamp / 1000, tz=UTC).strftime("%Y-%m-%d")
    else:
        date_str = f"Image {i+1}"
    
    # Choose visualization parameters
    vis_params = {
        "bands": ["B4", "B3", "B2"],  # RGB
        "min": 0,
        "max": 3000,
        "gamma": 1.4,
    }
    
    Map.addLayer(image, vis_params, name=date_str)

# Add a layer control and show the map
Map.centerObject(ee_parks, 12)
Map

Map(center=[32.35093667615046, -108.7165845437154], controls=(WidgetControl(options=['position', 'transparent_…

In [19]:
single_image = ee.Image(sentinel2.first())


In [26]:
# Keep only the reflectance bands (UInt16)
export_image = single_image.select([
    'B2', 'B3', 'B4']).clip(ee_parks).uint16()



In [25]:
# %% Export NDVI to Google Drive (Optional)
export_task_ndvi = ee.batch.Export.image.toDrive(
    image=export_image,
    description='Lordsburg Parks 2024-09-12',
    folder='sentinel',
    fileNamePrefix='lordsburg_parks_20240912',
    region=ee_parks,
    scale=10,
    crs='EPSG:32612',
    fileFormat='GeoTIFF',
    maxPixels=1e13
)
export_task_ndvi.start()